# AfyaFlow Pwani: Gemma 4 Stock-Out Early Warning Demo

AfyaFlow Pwani targets Track 3: Smart Health - AI-Driven Health Center & Supply Chain Management. The prototype focuses on one high-risk operational gap: medicine and diagnostic-test stock-outs that remain hidden in paper cards, phone calls, or fragmented reports until patients are already affected.

This notebook demonstrates the reproducible report-to-alert workflow with **synthetic data only**. On Kaggle GPU it loads the attached Keras Gemma 4 model input, extracts a structured stock report, then runs deterministic risk, approved tools, and bilingual handoff.

## Why Gemma 4 is central

Gemma 4 handles the unstructured part of the workflow: multilingual facility reports, strict JSON extraction, and (optionally) multimodal inputs. AfyaFlow keeps arithmetic and redistribution logic deterministic so the model cannot hallucinate operational decisions.

## 0. Kaggle setup

Before running:

1. Accelerator: **GPU** (T4 or better).
2. Add the **Keras Gemma 4** model input (your attached `gemma4` / V2 preset with `config.json`, `tokenizer.json`, and `model_*.weights.h5`).
3. Add the AfyaFlow repository either by cloning GitHub (Internet on) or attaching a dataset that contains `src/afyaflow`.

If the repo is not already on disk, uncomment the clone cell below.

In [ ]:
# Uncomment on a fresh Kaggle notebook if the repo is not already present:
# !git clone https://github.com/pinkleather221/afyaflow-pwani.git
# %cd afyaflow-pwani

import os
from pathlib import Path

# Prefer JAX on Kaggle GPU for KerasHub Gemma 4.
os.environ.setdefault("KERAS_BACKEND", "jax")

print("KERAS_BACKEND =", os.environ.get("KERAS_BACKEND"))

In [ ]:
!pip install -q -U "keras>=3.8" "keras-hub>=0.21"

import keras
import keras_hub

print("keras", keras.__version__)
print("keras_hub", keras_hub.__version__)
print("backend", keras.config.backend())

In [ ]:
from pathlib import Path
import os
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / "afyaflow-pwani",
    Path("/kaggle/working/afyaflow-pwani"),
    Path("/kaggle/working"),
    Path("/kaggle/input/afyaflow-pwani"),
]

ROOT = None
for path in candidates:
    if (path / "src" / "afyaflow" / "__init__.py").exists():
        ROOT = path.resolve()
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not find the afyaflow-pwani repository. "
        "Clone https://github.com/pinkleather221/afyaflow-pwani.git into /kaggle/working "
        "or attach a dataset that contains src/afyaflow."
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

print("Notebook root:", ROOT)
print("Working directory:", Path.cwd())

In [ ]:
from dataclasses import asdict

from afyaflow.data_loader import (
    load_facility_inventory,
    load_stock_report_examples,
    stock_report_from_expected,
)
from afyaflow.extraction import build_extraction_prompt, extract_with_rules
from afyaflow.gemma_client import GemmaClient
from afyaflow.risk_engine import calculate_stock_risk_with_transfers
from afyaflow.tool_registry import ToolCall, execute_tool_call
from afyaflow.workflow import run_report_workflow

inventory_path = ROOT / "data" / "synthetic" / "facility_inventory.json"
reports_path = ROOT / "data" / "synthetic" / "stock_reports.json"
inventory = load_facility_inventory(inventory_path)
examples = load_stock_report_examples(reports_path)
len(inventory), len(examples)

## 1. Locate and load the attached Gemma 4 Keras preset

Your Kaggle model input (`Gemma 4 · gemma4 · V2`) is a KerasHub preset directory. This cell finds any `/kaggle/input/...` folder that contains `config.json` + `model.weights.json` + `tokenizer.json`, then loads it with `Gemma4CausalLM.from_preset`.

In [ ]:
import json
import re
from typing import Any


def find_gemma4_preset(search_roots: list[Path] | None = None) -> Path:
    """Find a local KerasHub Gemma 4 preset under common Kaggle input paths."""

    roots = search_roots or [
        Path("/kaggle/input"),
        Path.cwd() / "models",
        Path.cwd(),
    ]

    preferred_markers = (
        "gemma4_instruct_2b",
        "gemma4_instruct_4b",
        "gemma4_instruct",
        "gemma4",
    )

    candidates: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        for config_path in root.rglob("config.json"):
            preset_dir = config_path.parent
            required = [
                preset_dir / "model.weights.json",
                preset_dir / "tokenizer.json",
                preset_dir / "task.json",
            ]
            if all(path.exists() for path in required):
                candidates.append(preset_dir)

    if not candidates:
        raise FileNotFoundError(
            "No Gemma 4 Keras preset found. Add the keras/gemma4 model as a notebook "
            "Input, then re-run this cell. Expected files include config.json, "
            "tokenizer.json, model.weights.json, and model_*.weights.h5."
        )

    def rank(path: Path) -> tuple[int, int, str]:
        text = str(path).lower()
        prefer = next(
            (index for index, marker in enumerate(preferred_markers) if marker in text),
            len(preferred_markers),
        )
        # Prefer instruct variants and shorter paths (likely the real preset root).
        instruct_penalty = 0 if "instruct" in text else 1
        return (prefer, instruct_penalty, str(path))

    candidates = sorted(set(candidates), key=rank)
    print("Found Gemma preset candidates:")
    for path in candidates:
        print(" -", path)
    return candidates[0]


GEMMA_PRESET = find_gemma4_preset()
print("Using preset:", GEMMA_PRESET)
print("Preset files:", sorted(p.name for p in GEMMA_PRESET.iterdir())[:20])

In [ ]:
print("Loading Gemma 4 from local Kaggle input (this can take a few minutes)...")
gemma_lm = keras_hub.models.Gemma4CausalLM.from_preset(
    str(GEMMA_PRESET),
    dtype="bfloat16",
)
print("Loaded:", type(gemma_lm).__name__)
print("Preset path:", GEMMA_PRESET)

## 2. Wire Gemma 4 into the AfyaFlow `ModelRuntime` boundary

The runtime asks Gemma for strict JSON, then AfyaFlow validates the payload with `StockReport` schemas. If parsing fails, the notebook falls back to deterministic rules so the rest of the demo still completes.

In [ ]:
class KaggleGemmaRuntime:
    """KerasHub Gemma 4 runtime that returns a JSON-compatible dict."""

    def __init__(self, model, *, max_length: int = 768) -> None:
        self.model = model
        self.max_length = max_length
        self.last_raw_text = ""

    def _wrap_prompt(self, prompt: str) -> str:
        return (
            "<|turn>user\n"
            f"{prompt}\n"
            "Return ONLY a single JSON object. No markdown fences. No commentary.<turn|>\n"
            "<|turn>model\n"
        )

    def _strip_prompt(self, output: Any, prompt: str) -> str:
        if isinstance(output, list):
            output = output[0]
        text = str(output)
        if text.startswith(prompt):
            return text[len(prompt) :]
        for marker in ("<|turn>model\n", "<start_of_turn>model\n"):
            index = text.rfind(marker)
            if index != -1:
                return text[index + len(marker) :]
        return text

    def _extract_json_object(self, text: str) -> dict[str, Any]:
        cleaned = text.strip()
        fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.DOTALL)
        if fenced:
            cleaned = fenced.group(1)
        else:
            match = re.search(r"(\{.*\})", cleaned, flags=re.DOTALL)
            if not match:
                raise ValueError(f"No JSON object found in model output: {text[:400]!r}")
            cleaned = match.group(1)
        payload = json.loads(cleaned)
        if not isinstance(payload, dict):
            raise ValueError("Model JSON must be an object")
        return payload

    def generate_json(self, prompt: str) -> dict[str, Any]:
        wrapped = self._wrap_prompt(prompt)
        output = self.model.generate({"prompts": [wrapped]}, max_length=self.max_length)
        raw = self._strip_prompt(output, wrapped)
        self.last_raw_text = raw
        print("Gemma raw output:\n", raw[:1200])
        return self._extract_json_object(raw)


runtime = KaggleGemmaRuntime(gemma_lm)
client = GemmaClient(model_name=str(GEMMA_PRESET), runtime=runtime)
print("GemmaClient ready with live Kaggle Gemma 4 runtime.")

## 3. Synthetic golden scenario

The golden demo starts with a mixed Swahili/English facility report about ORS sachets. Gemma extracts structured fields; AfyaFlow then computes risk and redistribution deterministically.

In [ ]:
example = examples[0]
raw_input = example["raw_input"]
print(raw_input)
print()
print(build_extraction_prompt(raw_input, "text"))

In [ ]:
print("Extracting with live Gemma 4...")
try:
    report = client.extract_stock_report(raw_input, source_type="text")
    extraction_mode = "gemma4"
except Exception as exc:
    print("Gemma extraction failed; using deterministic fallback.")
    print("Reason:", exc)
    report = extract_with_rules(raw_input, source_type="text")
    extraction_mode = "fallback"

print("Extraction mode:", extraction_mode)
asdict(report)

In [ ]:
risk = calculate_stock_risk_with_transfers(report, inventory)
asdict(risk)

## 4. Approved tool calling

Gemma is only allowed to request approved operational tools. The application validates each call before deterministic code executes it.

In [ ]:
handoff = execute_tool_call(
    ToolCall(name="draft_handoff_message", arguments={"language": "sw"}),
    report,
    inventory,
)
handoff

In [ ]:
result = run_report_workflow(
    raw_input,
    inventory_path=inventory_path,
    handoff_language="sw",
    runtime=runtime,
    model_name=str(GEMMA_PRESET),
)
{
    "facility": result["report"].facility,
    "item": result["report"].item,
    "risk_level": result["risk"].level,
    "days_of_stock": result["risk"].days_of_stock,
    "handoff": result["handoff"],
    "model_name": result["model_name"],
}

## 5. Mini evaluation with live Gemma extraction

This smoke evaluation runs the three public synthetic examples through Gemma 4 extraction and compares risk bands to the hand-labelled expectations.

In [ ]:
rows = []
for item in examples:
    expected_report = stock_report_from_expected(item)
    expected_risk = calculate_stock_risk_with_transfers(expected_report, inventory)

    mode = "gemma4"
    try:
        predicted_report = client.extract_stock_report(
            item["raw_input"],
            source_type=item.get("source_type", "text"),
        )
    except Exception as exc:
        print(f"{item['id']}: Gemma failed ({exc}); using fallback")
        predicted_report = extract_with_rules(
            item["raw_input"],
            source_type=item.get("source_type", "text"),
        )
        mode = "fallback"

    predicted_risk = calculate_stock_risk_with_transfers(predicted_report, inventory)
    rows.append(
        {
            "id": item["id"],
            "mode": mode,
            "facility": predicted_report.facility,
            "item": predicted_report.item,
            "expected_risk": item["expected"]["risk_level"],
            "actual_risk": predicted_risk.level,
            "risk_pass": item["expected"]["risk_level"] == predicted_risk.level,
            "facility_pass": predicted_report.facility == expected_report.facility,
            "item_pass": predicted_report.item == expected_report.item,
        }
    )

rows

In [ ]:
passed = sum(1 for row in rows if row["risk_pass"] and row["facility_pass"] and row["item_pass"])
print(f"Smoke evaluation: {passed}/{len(rows)} examples fully matched")
print("Preset used:", GEMMA_PRESET)
print("Extraction modes:", {row["id"]: row["mode"] for row in rows})

## Safety reminder

Gemma understands messy multilingual reports. AfyaFlow still controls the operational action through schema validation, human confirmation in the Streamlit app, deterministic risk arithmetic, an approved tool allowlist, and an append-only audit trail. Public demo data remains synthetic only.